# STEP 2: Feature Enrichment - Compute and Store Enhanced Features

**Purpose**: Compute all 37 enhanced features and store them in the database enrichments table

**What this notebook does**:
1. **Load Raw CVE Data** - Read cvss_vector, cwe, description from cves table
2. **Compute CVSS Features** - Parse CVSS vectors into 10 dimensional features
3. **Compute CWE Features** - Extract CWE intelligence (Top 25, categories, severity)
4. **Compute NLP Features** - Detect exploitation keywords in descriptions
5. **Compute Vendor Features** - Identify high-risk vendors
6. **Compute Interaction Features** - Create compound risk indicators
7. **Update Database** - Store all 37 computed features in enrichments table

**Input**: cves table (raw CVE data)
**Output**: enrichments table updated with 37 computed features

**Prerequisites**:
- STEP_1: Data ingestion completed
- STEP_2: External enrichments completed
- Database migration script executed (scripts/migrate_enrichments_schema.py)

---

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import warnings

import pandas as pd
import numpy as np
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

# Import project modules
from src.core.cve_database import CVEDatabase
from src.features.enhanced_features import EnhancedFeatureExtractor
from config.settings import settings

print(f"[OK] Project root: {project_root}")
print(f"[OK] Database: {settings.get_database_path()}")
print(f"[OK] Imports successful")
print(f"\n[INFO] This notebook will compute 37 enhanced features and store them in the database")

## 2. Database Connection & Validation

In [ ]:
# Connect to database
db = CVEDatabase()

# Get statistics
stats = db.get_statistics()

print("="*70)
print("DATABASE STATUS")
print("="*70)
print(f"Total CVEs: {stats['total_cves']:,}")
print(f"\nEnrichment Coverage:")
print(f"  KEV entries: {stats.get('kev_count', 0):,}")
print(f"  EPSS scores: {stats.get('epss_count', 0):,}")
print(f"  Healthcare flags: {stats.get('healthcare_count', 0):,}")
print(f"  ATT&CK mappings: {stats.get('attack_count', 0):,}")
print(f"  CHPL entries: {stats.get('chpl_count', 0):,}")
print("="*70)

## 3. Load Raw CVE Data for Feature Computation

In [ ]:
# Load CVEs with raw data needed for feature computation
query = """
SELECT 
    c.cve_id,
    c.cvss,
    c.cvss_vector,
    c.cwe,
    c.description,
    e.kev_flag,
    e.is_healthcare
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.cvss IS NOT NULL
ORDER BY c.cve_id
"""

print("[INFO] Loading CVE data for feature computation...")
df = pd.read_sql(query, db.conn)

print(f"\n[OK] Loaded {len(df):,} CVEs")
print(f"\nData availability:")
print(f"  CVEs with CVSS vector: {df['cvss_vector'].notna().sum():,} ({df['cvss_vector'].notna().mean()*100:.1f}%)")
print(f"  CVEs with CWE: {df['cwe'].notna().sum():,} ({df['cwe'].notna().mean()*100:.1f}%)")
print(f"  CVEs with description: {df['description'].notna().sum():,} ({df['description'].notna().mean()*100:.1f}%)")

# Display sample
print(f"\nSample data:")
df.head(3)

## 4. Initialize Feature Extractor

In [ ]:
# Initialize enhanced feature extractor
extractor = EnhancedFeatureExtractor()

print("[OK] EnhancedFeatureExtractor initialized")
print(f"\nFeature categories:")
print(f"  - CVSS Decomposition: 10 features")
print(f"  - CWE Intelligence: 8 features")
print(f"  - Description NLP: 10 features")
print(f"  - Vendor Features: 3 features")
print(f"  - Interaction Features: 6 features")
print(f"  Total: 37 computed features")

## 5. Compute Enhanced Features

This step computes all 37 features for each CVE. This may take several minutes for large datasets.

In [ ]:
print("="*70)
print("COMPUTING ENHANCED FEATURES")
print("="*70)
print(f"\n[INFO] Processing {len(df):,} CVEs...")
print(f"[INFO] This will compute 37 features per CVE")
print(f"[INFO] Estimated time: {len(df) / 1000:.1f} minutes (approximate)\n")

start_time = datetime.now()

# Apply enhanced feature extraction
enhanced_df = extractor.extract_all_features(df)

elapsed = (datetime.now() - start_time).total_seconds()

print(f"\n[OK] Feature computation completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"  Processing rate: {len(df)/elapsed:.0f} CVEs/second")
print(f"\n[INFO] Enhanced dataset shape: {enhanced_df.shape}")
print(f"  Total columns: {len(enhanced_df.columns)}")

# Show computed features sample
print(f"\nSample computed features:")
feature_cols = [c for c in enhanced_df.columns if c.startswith(('cvss_', 'cwe_', 'desc_', 'vendor_', 'ultimate', 'critical', 'network', 'auth', 'high_impact', 'healthcare_critical'))]
print(f"  Computed feature columns: {len(feature_cols)}")
enhanced_df[['cve_id'] + feature_cols[:5]].head(3)

## 6. Validate Computed Features

In [ ]:
print("="*70)
print("FEATURE VALIDATION")
print("="*70)

# Check for computed features
computed_features = {
    'CVSS Decomposition': ['cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s', 'cvss_c', 'cvss_i', 'cvss_a'],
    'CWE Intelligence': ['cwe_is_top25', 'cwe_is_injection', 'cwe_is_crypto'],
    'Description NLP': ['desc_has_rce', 'desc_has_sqli', 'desc_has_xss'],
    'Vendor Features': ['vendor_is_high_risk', 'vendor_is_healthcare'],
    'Interaction Features': ['ultimate_risk', 'critical_exploitable', 'network_accessible']
}

print(f"\nFeature Coverage:")
for category, features in computed_features.items():
    print(f"\n{category}:")
    for feat in features:
        if feat in enhanced_df.columns:
            non_null = enhanced_df[feat].notna().sum()
            pct = (non_null / len(enhanced_df)) * 100
            non_zero = (enhanced_df[feat] != 0).sum() if enhanced_df[feat].dtype in ['int64', 'float64'] else 0
            print(f"  {feat:30s}: {pct:5.1f}% coverage, {non_zero:6,} non-zero values")
        else:
            print(f"  {feat:30s}: [MISSING]")

print(f"\n" + "="*70)

## 7. Update Database with Computed Features

This step writes all computed features back to the enrichments table.

In [ ]:
print("="*70)
print("UPDATING DATABASE")
print("="*70)
print(f"\n[INFO] Preparing to update enrichments table with computed features")
print(f"[INFO] CVEs to update: {len(enhanced_df):,}")

# Prepare feature columns for database update
feature_columns = [
    # CVSS Decomposition
    'cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s', 
    'cvss_c', 'cvss_i', 'cvss_a', 'cvss_score_derived', 'cvss_severity_category',
    # CWE Intelligence
    'cwe_is_top25', 'cwe_is_injection', 'cwe_is_crypto', 'cwe_is_access_control',
    'cwe_is_input_validation', 'cwe_is_memory_corruption', 'cwe_category', 'cwe_severity_score',
    # Description NLP
    'desc_has_rce', 'desc_has_auth_bypass', 'desc_has_priv_esc', 'desc_has_sqli',
    'desc_has_xss', 'desc_has_dos', 'desc_has_buffer_overflow', 'desc_has_path_traversal',
    'desc_has_csrf', 'desc_has_xxe',
    # Vendor Features
    'vendor_is_high_risk', 'vendor_is_healthcare', 'vendor_risk_score',
    # Interaction Features
    'ultimate_risk', 'critical_exploitable', 'network_accessible', 
    'auth_not_required', 'high_impact_network', 'healthcare_critical'
]

# Filter to only include columns that exist in enhanced_df
available_features = [col for col in feature_columns if col in enhanced_df.columns]
print(f"[INFO] Available features to update: {len(available_features)} / {len(feature_columns)}")

if len(available_features) < len(feature_columns):
    missing = set(feature_columns) - set(available_features)
    print(f"[WARN] Missing features: {missing}")

In [ ]:
# Update database in batches
print(f"\n[INFO] Starting database update...")

batch_size = 1000
total_updated = 0
errors = 0

cursor = db.conn.cursor()

# Build UPDATE statement
set_clause = ", ".join([f"{col} = ?" for col in available_features])
update_sql = f"UPDATE enrichments SET {set_clause} WHERE cve_id = ?"

for i in tqdm(range(0, len(enhanced_df), batch_size), desc="Updating database"):
    batch = enhanced_df.iloc[i:i+batch_size]
    
    try:
        for _, row in batch.iterrows():
            values = [row.get(col) for col in available_features]
            values.append(row['cve_id'])  # Add cve_id for WHERE clause
            
            cursor.execute(update_sql, values)
            total_updated += 1
        
        # Commit batch
        db.conn.commit()
        
    except Exception as e:
        errors += 1
        print(f"\n[ERROR] Batch {i//batch_size + 1} failed: {e}")
        db.conn.rollback()

print(f"\n[OK] Database update completed")
print(f"  CVEs updated: {total_updated:,}")
print(f"  Errors: {errors}")
print(f"  Features per CVE: {len(available_features)}")

## 8. Verification - Query Updated Enrichments

In [ ]:
print("="*70)
print("VERIFICATION")
print("="*70)

# Query enrichments table to verify updates
verify_query = """
SELECT 
    cve_id,
    kev_flag,
    cvss_av,
    cvss_s,
    cwe_is_top25,
    desc_has_rce,
    vendor_is_high_risk,
    ultimate_risk,
    network_accessible
FROM enrichments
WHERE cvss_av IS NOT NULL
LIMIT 10
"""

verify_df = pd.read_sql(verify_query, db.conn)

print(f"\n[INFO] Sample of updated enrichments:")
print(verify_df)

# Check coverage
coverage_query = """
SELECT 
    COUNT(*) as total,
    SUM(CASE WHEN cvss_av IS NOT NULL THEN 1 ELSE 0 END) as has_cvss_av,
    SUM(CASE WHEN cwe_is_top25 IS NOT NULL THEN 1 ELSE 0 END) as has_cwe_features,
    SUM(CASE WHEN desc_has_rce IS NOT NULL THEN 1 ELSE 0 END) as has_nlp_features,
    SUM(CASE WHEN ultimate_risk IS NOT NULL THEN 1 ELSE 0 END) as has_interaction_features
FROM enrichments
"""

coverage_df = pd.read_sql(coverage_query, db.conn)

print(f"\n[STATS] Enrichment Coverage:")
total = coverage_df['total'].iloc[0]
for col in coverage_df.columns[1:]:
    count = coverage_df[col].iloc[0]
    pct = (count / total) * 100 if total > 0 else 0
    print(f"  {col:30s}: {count:6,} / {total:,} ({pct:5.1f}%)")

print(f"\n" + "="*70)

## 9. Summary & Next Steps

In [ ]:
print("="*70)
print("FEATURE ENRICHMENT SUMMARY")
print("="*70)
print(f"\n[OK] Feature enrichment completed successfully")
print(f"\n[STATS] Enrichment Results:")
print(f"  CVEs processed: {len(enhanced_df):,}")
print(f"  Features computed per CVE: {len(available_features)}")
print(f"  Database columns updated: {len(available_features)}")
print(f"  Processing time: {elapsed:.1f} seconds")

print(f"\n[ARCHITECTURE] Data Pipeline Status:")
print(f"  [OK] STEP_1: CVE Data Ingestion (raw NVD data)")
print(f"  [OK] STEP_2: External Enrichments (KEV, EPSS, Healthcare, ATT&CK, CHPL)")
print(f"  [OK] STEP_2: Feature Enrichment (37 computed features) <- CURRENT")
print(f"  [..] STEP_4: Feature Engineering + Labels")
print(f"  [..] STEP_5: Model Training + Scientific Protocol")
print(f"  [..] STEP_8: Advanced Models (artifact-aligned)")

print(f"\n[NEXT STEP] Run STEP_3_Feature_Engineering_Labels.ipynb")
print(f"\n[INFO] All 52 enrichment features are now available in the database:")
print(f"  - 15 external signals (KEV, EPSS, Healthcare, ATT&CK, CHPL, etc.)")
print(f"  - 37 computed features (CVSS, CWE, NLP, Vendor, Interactions)")
print(f"\n" + "="*70)

# Close database connection
db.conn.close()
print(f"\n[OK] Database connection closed")